# 🎙️ Kashmiri ASR Fine-Tuning: Meta Wav2Vec2-BERT 2.0 (`facebook/w2v-bert-2.0`)
**Dataset**: `munzirahangar/kas-news-dataset` (22,425 Kashmiri broadcast news utterances, 16 kHz)  
**Edition**: Google Colab Pro (A100 40GB/80GB & L4 24GB) — **Disconnect & Restart Resilient**

---

### 🛡️ Built-In Fault-Tolerance for Colab Interruptions
1. **Zero 30-Minute Startup Penalty on Restarts**: Uses fast Parquet text-column scanning (`<3s`) to build/load the Kashmiri CTC vocabulary (`vocab.json`) and deterministic `train`/`validation`/`test` split indices (`splits.json`), while decoding 16kHz audio and extracting 80-bin `SeamlessM4TFeatureExtractor` filterbanks **on-the-fly** inside PyTorch DataLoader workers.
2. **Step-Level Checkpointing + Corrupted Checkpoint Quarantine**: Saves checkpoints every `200` steps directly to Google Drive. If Colab disconnects *mid-save*, `find_latest_valid_checkpoint()` automatically quarantines the half-written checkpoint folder and resumes from the previous intact checkpoint.
3. **Continuous Best-Model Export**: Whenever validation `DA-WER` improves, a standalone copy of the model + processor is exported immediately to `<DRIVE_OUTPUT_DIR>/best_model`.
4. **Resumable Evaluation**: Test-set evaluation streams predictions batch-by-batch to `eval_predictions_test.jsonl` with `os.fsync()`. If disconnected at 80% during evaluation, re-running Cell 5 resumes from 80% and generates the full report.

In [ ]:
# Cell 1: Mount Google Drive, Clone/Pull Repo & Install Dependencies
import os
import subprocess
import torch
from google.colab import drive

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected! Go to Runtime -> Change runtime type and select A100 GPU or L4 GPU."
    )

# 1. Mount Google Drive for persistent checkpoints, splits, processor & evaluation logs
drive.mount("/content/drive")
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/nasr2_kashmiri_w2v_bert"
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
print(f"[✓] Persistent Google Drive directory: {DRIVE_OUTPUT_DIR}")

# 2. Clone or pull your repository into /content/nasr2
# Update REPO_URL if your repository URL differs:
REPO_URL = os.environ.get("NASR2_REPO_URL", "https://github.com/munzirds/nasr2.git")
REPO_DIR = "/content/nasr2"

if not os.path.exists(os.path.join(REPO_DIR, "pyproject.toml")):
    if os.path.exists("pyproject.toml"):
        REPO_DIR = os.getcwd()
    else:
        print(f"--> Cloning {REPO_URL} into {REPO_DIR}...")
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    print(f"--> Pulling latest changes in {REPO_DIR}...")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)

%cd {REPO_DIR}
!bash scripts/colab_setup.sh

In [ ]:
# Cell 2: Authenticate with Hugging Face (for private munzirahangar/kas-news-dataset)
import os
from getpass import getpass
from huggingface_hub import login
from kas_w2v_bert.data import resolve_hf_token

hf_token = resolve_hf_token()
if not hf_token:
    hf_token = getpass("Enter your Hugging Face token (or add HF_TOKEN to Colab Secrets): ").strip()

os.environ["HF_TOKEN"] = hf_token
login(token=hf_token, add_to_git_credential=False)
print("[✓] Hugging Face authentication verified.")

In [ ]:
# Cell 3: Inspect Existing Checkpoints / Run Status on Google Drive (Before Training/Resuming)
import json
import os
from kas_w2v_bert.checkpoint_manager import find_latest_valid_checkpoint, list_checkpoint_dirs_sorted

ckpts_dir = os.path.join(DRIVE_OUTPUT_DIR, "checkpoints")
status_file = os.path.join(DRIVE_OUTPUT_DIR, "run_status.json")

print("=" * 68)
print("  PERSISTENT CHECKPOINT & RUN STATUS DIAGNOSTICS")
print("=" * 68)
all_ckpts = list_checkpoint_dirs_sorted(ckpts_dir)
valid_ckpt = find_latest_valid_checkpoint(ckpts_dir, require_optimizer=True, quarantine_corrupted=False)
print(f"  Total Checkpoints on Drive: {len(all_ckpts)}")
print(f"  Latest Valid Checkpoint:    {valid_ckpt or 'None (Fresh Run)'}")

if os.path.isfile(status_file):
    with open(status_file, "r", encoding="utf-8") as f:
        st = json.load(f)
    print(f"  Last Recorded Status:       {st.get('status')}")
    print(f"  Global Step / Epoch:        step {st.get('global_step')} (epoch {st.get('epoch')})")
    print(f"  Best Validation DA-WER:     {st.get('best_da_wer')}%")
    print(f"  Last Updated (UTC):         {st.get('updated_at_utc')}")
print("=" * 68)

In [ ]:
# Cell 4: Run or Resume Wav2Vec2-BERT 2.0 Fine-Tuning
# Safe to re-run at any time after a Colab disconnect/restart — automatically resumes
# from the latest verified checkpoint in DRIVE_OUTPUT_DIR/checkpoints.
!python scripts/run_train.py \
    --config configs/w2v_bert_kas_news.yaml \
    --output_dir "{DRIVE_OUTPUT_DIR}" \
    --save_steps 200 \
    --eval_steps 200

In [ ]:
# Cell 5: Run Resumable Evaluation on Held-Out Test Split
# Writes batch predictions incrementally to DRIVE_OUTPUT_DIR/evaluation/eval_predictions_test.jsonl.
# If interrupted, re-running this cell resumes from the exact batch where it stopped.
!python scripts/run_eval.py \
    --config configs/w2v_bert_kas_news.yaml \
    --output_dir "{DRIVE_OUTPUT_DIR}" \
    --eval_split test

In [ ]:
# Cell 6: Display Markdown Evaluation Report & Sample Transcriptions
import os
from IPython.display import Markdown, display

report_md_path = os.path.join(DRIVE_OUTPUT_DIR, "evaluation", "eval_report_test.md")
if os.path.isfile(report_md_path):
    with open(report_md_path, "r", encoding="utf-8") as f:
        display(Markdown(f.read()))
else:
    print(f"Evaluation report not found at {report_md_path}. Run Cell 5 first.")